# COMP7103 Lab 11 · Recommend movies two ways

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doubledaibo/COMP7103B/blob/main/labs/lab11/Lab11_recommenders.ipynb)

A film website wants to show each user 10 films to watch next. You build two recommenders from the lecture on the same MovieLens ratings: **item-based collaborative filtering** (films rated alike by the same users) and an **embedding recommender** (films whose descriptions are close to the films the user liked). Then you give both **one test**: train on the ratings before a cut-off date, and check the 10 recommendations against the films each user watched after it.

| Time | What you do |
|---|---|
| 5 min | Brief |
| 30 min | Guided work: two recommenders, one test |
| 10 min | Share-out |

This lab is ungraded practice for the final exam. Data: MovieLens ml-latest-small (GroupLens Research), used under the GroupLens terms of use; Harper and Konstan (2015), *The MovieLens Datasets: History and Context*, ACM TiiS 5(4). The data is not shipped with the lab: the notebook downloads it from GroupLens.

**Using this notebook in Colab.** Choose *File → Save a copy in Drive* first, and work in your copy. Run a cell with **Shift + Enter**. Run the cells from top to bottom. Where you see `___`, replace it with your own code, then run the cell again.

## 0 · Setup

This cell loads the libraries and defines two helpers: `get()` finds the files of this lab, and `movielens()` downloads the MovieLens ratings from GroupLens (about 1 MB) the first time you use it.

In [ ]:
import os, time, zipfile, difflib, warnings, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
warnings.filterwarnings("ignore")

# A blank (___) that you have not filled in stops with: name '___' is not defined.
# (The next two lines make sure of that. You do not need to read or change them.)
get_ipython().displayhook.cache_size = 0
globals().pop("___", None)

REPO = "https://raw.githubusercontent.com/doubledaibo/COMP7103B/main/labs/lab11/data/"

# get(name) gives the path of a data file: the copy next to the notebook if there is one, otherwise a fresh
# download from the course GitHub. You do not need to read how it works.
def get(name):
    """Use the local file if it is there (Lab 11 downloaded from GitHub), otherwise download it."""
    for path in [name, os.path.join("data", name),]:
        if os.path.exists(path):
            return path
    os.makedirs("data", exist_ok=True)
    try:
        urllib.request.urlretrieve(REPO + name, os.path.join("data", name))
    except Exception:
        raise SystemExit(f"Could not download {name}. Upload it with the folder icon on the left, "
                         "or download the Lab 11 from GitHub: https://github.com/doubledaibo/COMP7103B/tree/main/labs/lab11")
    return os.path.join("data", name)

ML_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"

def movielens(name):
    """Path of a MovieLens file: the folder ml-latest-small/ if it is here, otherwise download the zip from GroupLens."""
    if not os.path.exists(os.path.join("ml-latest-small", "tags.csv")):
        try:
            urllib.request.urlretrieve(ML_URL, "ml-latest-small.zip")
            zipfile.ZipFile("ml-latest-small.zip").extractall(".")
        except Exception as e:
            raise SystemExit(f"Could not download MovieLens ({e}). Download ml-latest-small.zip from "
                             "https://grouplens.org/datasets/movielens/latest/, unzip it next to this notebook "
                             "(it makes a folder ml-latest-small/) and run this cell again.")
    return os.path.join("ml-latest-small", name)

## 1 · The data and the time split (5 min)

MovieLens ml-latest-small has 100,836 ratings (0.5 to 5 stars) by 610 users of 9,742 films, from 1996 to 2018. We keep the **1,297 films with at least 20 ratings**: they are the catalogue $I$ in this lab.

In [ ]:
ratings = pd.read_csv(movielens("ratings.csv"))       # userId, movieId, rating (0.5 to 5 stars), timestamp
movies = pd.read_csv(movielens("movies.csv")).set_index("movieId")   # title, genres; look up a film by its id
tags = pd.read_csv(movielens("tags.csv"))             # userId, movieId, tag, timestamp
print(f"MovieLens: {len(ratings):,} ratings by {ratings['userId'].nunique()} users; {len(movies):,} films; {len(tags):,} tags")

MIN_RATINGS = 20                                      # keep films with at least 20 ratings
counts = ratings["movieId"].value_counts()            # movieId -> number of ratings
FILMS = np.sort(counts[counts >= MIN_RATINGS].index.values)   # the catalogue I of this lab (film ids, sorted)
FILM_SET = set(FILMS)
sub = ratings[ratings["movieId"].isin(FILM_SET)]      # the ratings of these films only

def nice_title(t):
    """'Matrix, The (1999)' -> 'The Matrix (1999)'."""
    t = " ".join(t.split())                           # single spaces
    # Look for ", The", ", A" or ", An", starting from the end of the title
    for pos in range(len(t) - 1, -1, -1):
        if t[pos:pos + 2] != ", ":
            continue
        for article in ["The", "A", "An"]:
            rest = t[pos + 2 + len(article):]         # what follows the article
            if t[pos + 2:pos + 2 + len(article)] != article:
                continue
            # the article must end the title, or be followed by a space and a bracket, e.g. " (1999)"
            if rest == "" or (rest[0].isspace() and rest.lstrip().startswith("(")):
                return article + " " + t[:pos] + rest
    return t

print(f"Subset: {len(FILMS):,} films, {len(sub):,} ratings by {sub['userId'].nunique()} users")

**Split by time.** A random split would leak the future into training. We train on every rating before **1 October 2016** and test on the films each user rated after it. A rating is our sign that the user watched the film. We test the users with at least 10 ratings and at least 5 liked films (4 stars or more) before the cut-off, and at least 5 films after it. $T(u)$ is the set of films user $u$ rated after the cut-off.

Fill in the condition that keeps the ratings before the cut-off.

In [ ]:
CUT = pd.Timestamp("2016-10-01").timestamp()          # the cut-off, 1 October 2016, in Unix time (seconds since 1970)
train = sub[___]                       # ratings before the cut-off
test = sub[sub["timestamp"] >= CUT]                   # ratings after it

n_train = train.groupby("userId").size()                          # ratings per user before the cut-off
n_liked = train[train["rating"] >= 4].groupby("userId").size()    # liked films (4 stars or more) per user
n_test = test.groupby("userId").size()                            # ratings per user after the cut-off
TEST_USERS = []
for u in n_test.index:
    # .get(u, 0): 0 for a user with no ratings of that kind
    if n_test[u] >= 5 and n_train.get(u, 0) >= 10 and n_liked.get(u, 0) >= 5:
        TEST_USERS.append(u)
TEST_USERS = sorted(TEST_USERS)

T = {}                                                # T(u): the films u rated after the cut-off
for u in TEST_USERS:
    T[u] = set(test[test["userId"] == u]["movieId"])

sizes = [len(T[u]) for u in TEST_USERS]
print(f"train: {len(train):,} ratings · test: {len(test):,} ratings · test users: {len(TEST_USERS)} "
      f"(median {int(np.median(sizes))} test films each)")

**The user–item matrix** $R$: one row per user, one column per film, training ratings only, 0 where a user did not rate a film.

In [ ]:
USERS = np.sort(train["userId"].unique())
row_of = {}                                           # user id -> row of R
for i in range(len(USERS)):
    row_of[USERS[i]] = i
col_of = {}                                           # film id -> column of R
for j in range(len(FILMS)):
    col_of[FILMS[j]] = j

R = np.zeros((len(USERS), len(FILMS)))                # a missing rating counts as 0
rows = train["userId"].map(row_of)                    # the row of each training rating
cols = train["movieId"].map(col_of)                   # its column
R[rows, cols] = train["rating"].values                # put every training rating in its cell
empty = 1 - (R > 0).mean()                            # the share of cells with no rating
print(f"R: {R.shape[0]} users x {R.shape[1]:,} films; {empty:.1%} of the cells are empty")

def title(m):
    return nice_title(movies["title"][m])

## 2 · Item-based collaborative filtering (10 min)

Find films rated like this film, then predict from the user's own ratings of them (Lecture 11, *Item-based CF*).

Cosine similarity between film columns (sums over all users; a missing rating counts as 0):

$$\mathrm{sim}(i,j)=\frac{\sum_u r_{u,i}\,r_{u,j}}{\sqrt{\sum_u r_{u,i}^2}\,\sqrt{\sum_u r_{u,j}^2}}$$

Fill in the axis: we need one length per film, that is, per **column** of $R$.

In [ ]:
norms = np.sqrt((R ** 2).sum(axis=___))                # the length of each film column
norms[norms == 0] = 1          # a film nobody rated before the cut-off: similarity 0 to every film
S = (R.T @ R) / np.outer(norms, norms)                 # S[i, j] = sim(i, j)
np.fill_diagonal(S, 0)                                 # a film is not its own neighbour
print("S:", S.shape, "· films with no ratings before the cut-off:", int((R.sum(axis=0) == 0).sum()))

A quick check: the films most similar to *Toy Story* (1995).

In [ ]:
toy = col_of[1]                                        # movieId 1 is Toy Story (1995)
order = np.argsort(-S[toy])[:5]                        # the 5 film columns with the highest similarity
films = []
sims = []
for j in order:
    films.append(title(FILMS[j]))
    sims.append(round(S[toy, j], 3))
pd.DataFrame({"film": films, "sim": sims})

**The prediction** for user $u$ and a film $i$ that $u$ has not rated:

$$\hat r_{u,i}=\frac{\sum_{j\in N_k(i;u)} \mathrm{sim}(i,j)\, r_{u,j}}{\sum_{j\in N_k(i;u)} |\mathrm{sim}(i,j)|}$$

$N_k(i;u)$ = the $k$ films most similar to $i$ that $u$ rated, with sim > 0. We use $k = 20$ neighbours (a different k from the 10 in precision@10 later). The recommendation is the 10 unseen films with the highest $\hat r_{u,i}$. Fill in the numerator.

Hint: `r` holds u's ratings, one per column of $R$; `rated` holds the columns of the films u rated; `nb` holds the positions in `rated` of film $i$'s $k$ neighbours, in the same order as `s`.

In [ ]:
K = 20                                                 # neighbours per film

def predict_cf(u):
    """Predicted rating r_hat(u, i) for every film i; films u already rated get -inf."""
    r = R[row_of[u]]                                   # u's ratings (0 = not rated)
    rated = np.flatnonzero(r)                          # the columns of the films u rated before the cut-off
    pred = np.zeros(len(FILMS))
    for i in range(len(FILMS)):
        sims = S[i, rated]                             # sim(i, j) for every film j that u rated
        nb = np.argsort(-sims)[:K]                     # N_k(i; u): positions (in rated) of the K most similar
        s = sims[nb]                                   # their similarities
        s[s < 0] = 0                                   # keep the neighbours with sim > 0 (the others count 0)
        num = (s * ___).sum()                          # sum of sim(i, j) * r_uj
        den = np.abs(s).sum()                          # sum of |sim(i, j)|
        if den > 0:
            pred[i] = num / den
    pred[rated] = -np.inf                              # never recommend a film u has already rated
    return pred

def top10(scores):
    """The 10 films with the highest scores, best first."""
    order = np.argsort(-scores, kind="stable")         # film columns, highest score first
    return FILMS[order[:10]]

def recommend_cf(u):
    return top10(predict_cf(u))

u = TEST_USERS[0]
liked = train[(train["userId"] == u) & (train["rating"] >= 4)]["movieId"]
liked_titles = [title(m) for m in liked[:8]]
print(f"User {u} liked:", "; ".join(liked_titles), "...")
cf_titles = [title(m) for m in recommend_cf(u)]
print("Item-based CF recommends:", "; ".join(cf_titles))

## 3 · The embedding recommender (8 min)

MovieLens has no plot summaries, so each film's description is its title, year, genres and the tags users gave it **before the cut-off** (tags from the test period would leak the future). A sentence-embedding model, all-MiniLM-L6-v2, turns each description into a vector of 384 numbers; films with similar descriptions get vectors that point the same way.

In [ ]:
train_tags = tags[(tags["timestamp"] < CUT) & tags["movieId"].isin(FILM_SET)]   # tags given before the cut-off
tag_text = {}                                          # film id -> its up to 8 most used tags
for m, group in train_tags.groupby("movieId"):         # one group of rows per film
    words = group["tag"].str.lower().str.strip()
    most_used = words.value_counts().index[:8]         # the 8 most used tags, most used first
    tag_text[m] = ", ".join(most_used)

def describe(m):
    d = f"{title(m)}. Genres: {movies['genres'][m].replace('|', ', ')}."
    if m in tag_text:
        d = d + f" Tags: {tag_text[m]}."
    return d

DESC = [describe(m) for m in FILMS]                    # one description per film, in the order of FILMS
print(f"{len(DESC):,} descriptions; {len(tag_text)} films have tags. Examples:")
for i in [0, 300, 900]:
    print(DESC[i])

Embed all descriptions (about a minute on a Colab CPU). `normalize_embeddings=True` makes every vector length 1, so the dot product of two film vectors is their cosine similarity.

In [ ]:
encoder = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
t0 = time.time()
E = encoder.encode(___, batch_size=64, normalize_embeddings=True)   # one row per film, in the order of FILMS
print("E:", E.shape, f"· {time.time() - t0:.0f} s")

**The user profile** is the mean of the vectors of the films $u$ liked (rated 4 stars or more) before the cut-off. Score every film by its dot product with the profile and recommend the 10 best unseen films. (The profile does not have length 1, but every score is divided by the same number, so the order is the order of cosine similarity.)

In [ ]:
def recommend_emb(u):
    r = R[row_of[u]]
    liked = np.flatnonzero(r >= ___)                   # the columns of the films u rated 4 stars or more
    profile = E[liked].mean(axis=0)                    # the mean of their vectors (axis=0: down the rows)
    scores = E @ ___                                   # one score per film
    scores[r > 0] = -np.inf                            # never recommend a film u has already rated
    return top10(scores)

emb_titles = [title(m) for m in recommend_emb(u)]
print("Embeddings recommend:", "; ".join(emb_titles))

## 4 · One test: precision@10 and coverage (7 min)

$L_k(u)$ = the top $k$ films recommended to $u$; $T(u)$ = the films $u$ rated after the cut-off; $|L_k(u)\cap T(u)|$ = the hits.

$$\text{Precision@}k=\frac{|L_k(u)\cap T(u)|}{k}\ \text{(averaged over the test users)}\qquad \text{Coverage}=\frac{|\bigcup_u L_k(u)|}{|I|}$$

Coverage is the share of the catalogue that appears in at least one user's top 10. We compute it over the test users. As a reference we add the simplest recommender: the most popular films (the most ratings before the cut-off) that the user has not rated. Fill in the two denominators.

In [ ]:
def precision_at_k(lists, k=10):
    """Mean over users of |L_k(u) & T(u)| / k; lists = {user: recommended films, best first}."""
    values = []
    for u in lists:
        top = set(lists[u][:k])                        # L_k(u)
        hits = len(top & T[u])                         # & = the films in both sets
        values.append(hits / ___)
    return float(np.mean(values))

def films_shown(lists):
    """All films that appear in at least one user's list."""
    shown = set()
    for u in lists:
        shown = shown | set(lists[u])                  # | = the films in either set
    return shown

def coverage(lists):
    """Share of the catalogue I that appears in at least one user's list."""
    shown = films_shown(lists)
    return len(shown) / ___

popular = train["movieId"].value_counts().index.values    # film ids, most ratings before the cut-off first

def recommend_popular(u):
    seen = set(FILMS[R[row_of[u]] > 0])                # the films u rated before the cut-off
    top = []
    for m in popular:
        if m not in seen:
            top.append(m)
        if len(top) == 10:
            break
    return np.array(top)

RECOMMENDERS = {"Most popular (reference)": recommend_popular, "Item-based CF": recommend_cf, "Embeddings": recommend_emb}
LISTS = {}                                             # recommender name -> {user: its 10 films}
for name in RECOMMENDERS:
    recommend = RECOMMENDERS[name]                     # a function: user -> 10 films
    LISTS[name] = {}
    for u in TEST_USERS:
        LISTS[name][u] = recommend(u)

rows = {}
for name in LISTS:
    rows[name] = {"precision@10": round(precision_at_k(LISTS[name]), 3),
                  "coverage (%)": round(100 * coverage(LISTS[name]), 1),
                  "different films shown": len(films_shown(LISTS[name]))}
results = pd.DataFrame.from_dict(rows, orient="index")   # one row per recommender
results

Each list has 10 films, so 25 test users can be shown at most 250 different films (19% of the catalogue). Compare the recommenders with each other, not with 100%. With 25 users and 10 films each, one hit changes precision@10 by 1/250 = 0.004, so a difference of one or two hits is a tie.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))       # two bar charts side by side
colours = ["#b0b0b0", "#6d4fc2", "#3a5a8c"]
ax[0].bar(results.index, results["precision@10"], color=colours)
ax[0].set_title("Precision@10 (test period)")
ax[1].bar(results.index, results["coverage (%)"], color=colours)
ax[1].set_title("Coverage (% of the 1,297 films)")
for a in ax:
    a.bar_label(a.containers[0], fmt="%.3g", fontsize=8)   # write the value on each bar
    a.tick_params(axis="x", labelsize=8)
plt.tight_layout()
plt.show()

Look at one user: the 10 films each recommender shows, with ✓ for a film the user rated after the cut-off. Change the 1 in `TEST_USERS[1]` (0 to 24) to see another user.

In [ ]:
u = TEST_USERS[1]
print(f"User {u}: {len(T[u])} films rated after the cut-off")
table = {}
for name in LISTS:
    column = []
    for m in LISTS[name][u]:
        if m in T[u]:
            column.append("✓ " + title(m))             # the user rated this film after the cut-off
        else:
            column.append("  " + title(m))
    table[name] = column
pd.DataFrame(table)

## 5 · Optional: an LLM recommender (if you have time)

1. Run the next cell. It prints a message with the films that 5 test users liked before the cut-off and a list of 150 candidate films (the most rated films before the cut-off).
2. Paste the message into any chat LLM you use. The message holds film titles and MovieLens user numbers only, no personal data.
3. Paste the LLM's whole answer between the triple quotes in the cell after.

No LLM available? Use the ready-made answer (see the comment in that cell). It comes from Qwen2.5-1.5B-Instruct, a small open model.

In [ ]:
prompt = open(get("llm_prompt.txt")).read()
print(prompt)

In [ ]:
llm_reply = """
paste the LLM's answer here
"""
# No LLM? Use the ready-made answer instead:
# llm_reply = open(get("llm_fallback.txt")).read()
print(llm_reply)

The LLM answers with titles, not film ids. The next cell reads its 10 titles per user and matches each title to MovieLens (ignoring case, punctuation, a leading *The* and the year, and allowing small spelling differences). Then it checks the answer against the catalogue, as the lecture asks: titles it could not find, films outside the candidate list, and films the user had already rated.

In [ ]:
# The 5 users in the message: the lines "User 15 liked ..."
LLM_USERS = []
for line in prompt.splitlines():
    words = line.split()
    if len(words) >= 3 and words[0] == "User" and words[1].isdigit() and words[2].startswith("liked"):
        LLM_USERS.append(int(words[1]))

# The candidate titles: the lines after "Candidate films", up to the next empty line
after = prompt.split("Candidate films")[1]
block = after.split("\n\n")[0]
CANDIDATES = block.splitlines()[1:]                    # one title per line (the first line is the heading)

def year(t):
    """The last year written in brackets, e.g. "Heat (1995)" -> "1995"; None if there is none."""
    found = None
    for i in range(len(t) - 5):
        if t[i] == "(" and t[i + 5] == ")" and t[i + 1:i + 5].isdigit():
            found = t[i + 1:i + 5]
    return found

def key(t):
    """A matching key: lower case, letters and digits only, no year, no leading The/A/An, no other title in brackets."""
    t = nice_title(t).lower()
    while "(" in t and ")" in t[t.index("("):]:        # remove every "( ... )" and the spaces before it
        start = t.index("(")
        end = t.index(")", start)
        t = t[:start].rstrip() + t[end + 1:]
    t = t.strip()
    for article in ["the ", "a ", "an "]:
        if t.startswith(article):
            t = t[len(article):]
            break
    return "".join([ch for ch in t if ch in "abcdefghijklmnopqrstuvwxyz0123456789"])

IDS = {}                                               # key -> MovieLens ids (all 9,742 films)
for m in movies.index:
    k = key(movies["title"][m])
    if k not in IDS:
        IDS[k] = []
    IDS[k].append(m)

def match(t):
    """The MovieLens id of a title written by the LLM, or None if there is no close title."""
    k = key(t)
    if k not in IDS:
        close = difflib.get_close_matches(k, list(IDS), n=1, cutoff=0.9)    # allow small spelling differences
        if len(close) == 0:
            return None
        k = close[0]
    # Several films can share a key: prefer the same year, then a film in the catalogue I
    for want_same_year in [True, False]:
        for want_in_catalogue in [True, False]:
            for m in IDS[k]:
                same_year = year(movies["title"][m]) == year(t)
                in_catalogue = m in FILM_SET
                if same_year == want_same_year and in_catalogue == want_in_catalogue:
                    return m

def user_heading(line):
    """'User 15: Heat (1995); Up (2009)' -> (15, ': Heat (1995); Up (2009)'); None if the line is not a user heading."""
    k = 0
    while k < len(line) and not (line[k].isalnum() or line[k] == "_"):
        k = k + 1                                      # skip symbols at the start, such as "#" or "-"
    rest = line[k:]
    if rest[:3].lower() == "for" and rest[3:4].isspace():
        rest = rest[3:].lstrip()                       # "For User 15"
    if rest[:4].lower() != "user":
        return None
    rest = rest[4:].lstrip()
    n = 0
    while n < len(rest) and rest[n].isdigit():
        n = n + 1                                      # the digits of the user number
    if n == 0:
        return None
    after = rest[n:]
    if after[:1].isalnum() or after[:1] == "_":
        return None                                    # e.g. "User 15a" is not a user number
    return int(rest[:n]), after

def list_item(line):
    """'3. Heat (1995)' or '- Heat (1995)' -> 'Heat (1995)'; None if the line is not a list item."""
    if line.startswith("-"):
        return line[1:].lstrip()
    n = 0
    while n < len(line) and line[n].isdigit():
        n = n + 1
    if n > 0 and n < len(line) and line[n] in ".)":
        return line[n + 1:].lstrip()
    return None

def cut_after_year(t):
    """'Heat (1995) - a great heist film' -> 'Heat (1995)'."""
    for i in range(len(t) - 5):
        if t[i] == "(" and t[i + 5] == ")" and t[i + 1:i + 5].isdigit():
            return t[:i + 6]
    return t

def parse_reply(text):
    """{user: [titles]} from 'User 15:' lines followed by numbered or bulleted titles."""
    out = {}
    u = None                                           # the user whose list we are reading
    for line in text.splitlines():
        line = line.strip()
        if line[:1] in ("*", "•", "-") and line[1:2].isspace():
            line = "- " + line[1:].strip()             # every bullet becomes "- "
        line = line.replace("*", "").strip()           # drop bold and italic marks
        head = user_heading(line)
        if head is not None:
            u = head[0]
            out[u] = []
            for part in head[1].split(";"):            # titles on the heading line itself
                if year(part) is not None:
                    out[u].append(part.strip())
        elif u is not None:
            t = list_item(line)
            if t is not None:
                out[u].append(cut_after_year(t).strip(' "_'))   # drop a reason after the year
    return out

titles = parse_reply(llm_reply)
cand_ids = set()
for t in CANDIDATES:
    cand_ids.add(match(t))

LLM = {}                                               # user -> the LLM's films in the catalogue I
rows = []
for u in LLM_USERS:
    ids = []
    for t in titles.get(u, [])[:10]:                   # its first 10 titles for this user
        ids.append(match(t))
    seen = set(FILMS[R[row_of[u]] > 0])                # films u rated before the cut-off
    LLM[u] = [m for m in ids if m in FILM_SET]
    not_candidate = 0
    already_rated = 0
    for m in ids:
        if m is not None and m not in cand_ids:
            not_candidate = not_candidate + 1
        if m in seen:
            already_rated = already_rated + 1
    rows.append({"user": u, "titles": len(ids), "not found": ids.count(None),
                 "not in candidate list": not_candidate, "already rated": already_rated,
                 "hits": len(set(ids) & T[u])})
pd.DataFrame(rows).set_index("user")

Score the LLM's lists the same way, and compare with the other recommenders **on the same 5 users** (coverage over 5 users is small for every recommender).

In [ ]:
five = {}                                              # the same lists, for the 5 LLM users only
for name in LISTS:
    five[name] = {}
    for u in LLM_USERS:
        five[name][u] = LISTS[name][u]
five["LLM (optional)"] = ___
rows = {}
for name in five:
    rows[name] = {"precision@10": round(precision_at_k(five[name]), 3),
                  "coverage (%)": round(100 * coverage(five[name]), 1)}
pd.DataFrame.from_dict(rows, orient="index")

## 6 · Share-out (10 min)

- Report **precision@10** and **coverage** for item-based CF and for the embedding recommender (the table in section 4), and for the LLM if you tried it (on its 5 users); we collect them on the class board.
- Which recommender is more accurate, and which covers more of the catalogue? Where does the most-popular reference sit?
- A low precision@10 is a cautious estimate: a film the user did not rate after the cut-off was not watched *here*, which does not prove the user would dislike it.
- The habit from the lecture: every recommender, including an LLM, is scored on the same users with the same time split.